# KPI_Design

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

order_items_date_columns = ['shipping_limit_date']

orders_master = pd.read_csv('../data/processed/orders_master.csv', parse_dates=date_cols)
items_master = pd.read_csv('../data/processed/items_master.csv', parse_dates=order_items_date_columns)
customers_cleaned   = pd.read_csv('../data/interim/olist_customers_cleaned.csv')


print(orders_master.shape)
print(items_master.shape)

(99441, 23)
(112650, 19)


## Sales KPIs

In [3]:
# Total Orders 
total_orders = orders_master['order_id'].nunique()
total_orders

99441

In [4]:
# Total Items Sold 
total_items_sold = items_master.shape[0]
total_items_sold

112650

In [5]:
# Total Sales
total_sales = orders_master['total_item_price'].sum()
total_sales

np.float64(13591643.7)

In [6]:
# AOV (Avrage Order Vales)
aov = orders_master['total_item_price'].mean()
aov_with_freight = (orders_master['total_item_price'] + orders_master['total_freight_value']).mean()

print(f"KPI 4 — Average Order Value (items only): R$ {aov:.2f}")
print(f"        Average Order Value (incl. freight): R$ {aov_with_freight:.2f}")

KPI 4 — Average Order Value (items only): R$ 137.75
        Average Order Value (incl. freight): R$ 160.58


## Customer KPIs

In [7]:
# Total Cosumers
total_customers = customers_cleaned['customer_unique_id'].nunique()
total_customers

96096

In [19]:
# Customer Repeat Rate
grouped = customers_cleaned.groupby('customer_unique_id')['customer_id'].size()
repeat_customers = (grouped > 1).sum()

repeat_customers_rate = repeat_customers / total_customers * 100
print(f"KPI 7 — Customer Repeat Rate: {repeat_customers_rate:.2f}%")
print(f"Base population: {total_customers} unique customers ({repeat_customers} with >1 order)")

KPI 7 — Customer Repeat Rate: 3.12%
Base population: 96096 unique customers (2997 with >1 order)


## Delivery KPIs

In [9]:
# On-Time Delivery Rate
delivered = orders_master[orders_master['order_delivered_customer_date'].notna()]

on_time_rate = delivered['is_on_time'].mean() * 100

print(f"KPI 1 — On-Time Delivery Rate: {on_time_rate:.2f}%")
print(f"Base population: {len(delivered)} delivered orders (out of {len(orders_master)} total)")

KPI 1 — On-Time Delivery Rate: 91.89%
Base population: 96476 delivered orders (out of 99441 total)


In [10]:
# Avrage Delivery Time 
avg_delivery_hours = delivered['purchase_to_customer_hours'].mean()
median_delivery_hours = delivered['purchase_to_customer_hours'].median()

print(f"KPI 2 — Avg. Delivery Time: {avg_delivery_hours/24:.1f} days (mean)")
print(f"                             {median_delivery_hours/24:.1f} days (median)")

KPI 2 — Avg. Delivery Time: 12.6 days (mean)
                             10.2 days (median)


In [20]:
delivered_valid = orders_master[
    (orders_master['order_delivered_customer_date'].notna()) &
    (~orders_master['has_invalid_duration'])
]

is_late_rate = delivered_valid['is_late'].mean() * 100
print(f"KPI 9 — Late Delivery Rate: {is_late_rate:.2f}%")
print(f"Base population: {len(delivered_valid)} delivered orders with valid duration (out of {len(orders_master)} total)")

KPI 9 — Late Delivery Rate: 6.85%
Base population: 95103 delivered orders with valid duration (out of 99441 total)


## Customer Satisfaction KPIs

In [12]:
# Average Review Score
avg_review_score = orders_master['review_score'].mean()
n_reviews = orders_master['review_score'].notna().sum()

print(f"KPI 3 — Avg. Review Score: {avg_review_score:.2f} / 5")
print(f"Based on {n_reviews} orders with a review ({n_reviews/len(orders_master)*100:.1f}% of all orders)")

KPI 3 — Avg. Review Score: 4.09 / 5
Based on 98673 orders with a review (99.2% of all orders)


In [21]:
low_rating_rate = orders_master["review_score"].le(2).mean() * 100
print(f"KPI 12 — Low Rating Rate (1-2 stars): {low_rating_rate:.2f}%")
print(f"Base population: {orders_master['review_score'].notna().sum()} orders with a review")

KPI 12 — Low Rating Rate (1-2 stars): 14.58%
Base population: 98673 orders with a review


## Seller KPIs

In [22]:
# Total Sellers
total_seller = items_master['seller_id'].nunique()
print(f"KPI 13 — Total Sellers: {total_seller}")

# Seller Concentration (Top 10% revenue share)
seller_revenue = items_master.groupby('seller_id')['price'].sum().sort_values(ascending=False)
total_revenue = seller_revenue.sum()

top10pct_n = int(len(seller_revenue) * 0.10)
top10pct_share = seller_revenue.iloc[:top10pct_n].sum() / total_revenue * 100

print(f"KPI 14 — Seller Concentration: Top 10% of sellers ({top10pct_n} sellers) generate {top10pct_share:.2f}% of total revenue")

KPI 13 — Total Sellers: 3095
KPI 14 — Seller Concentration: Top 10% of sellers (309 sellers) generate 67.49% of total revenue


In [25]:
# KPI Dictionary


kpi_dictionary = [
    {
        "KPI": "Total Orders",
        "Grain": "Order-level",
        "Base Population": "All orders, no filter",
        "Formula": "orders_master['order_id'].nunique()",
        "Value": total_orders
    },
    {
        "KPI": "Total Items Sold",
        "Grain": "Item-level",
        "Base Population": "All items sold",
        "Formula": "items_master.shape[0]",
        "Value": total_items_sold
    },
    {
        "KPI": "Total Sales (R$)",
        "Grain": "Order-level (aggregated from item)",
        "Base Population": "Sum of total_item_price across all orders",
        "Formula": "orders_master['total_item_price'].sum()",
        "Value": total_sales
    },
    {
        "KPI": "Average Order Value (R$)",
        "Grain": "Order-level",
        "Base Population": "All orders",
        "Formula": "orders_master['total_item_price'].mean()",
        "Value": aov
    },
    {
        "KPI": "AOV incl. Freight (R$)",
        "Grain": "Order-level",
        "Base Population": "All orders",
        "Formula": "(total_item_price + total_freight_value).mean()",
        "Value": aov_with_freight
    },
    {
        "KPI": "Total Unique Customers",
        "Grain": "Customer-level (customer_unique_id)",
        "Base Population": "All unique customers (not per-order ID)",
        "Formula": "customers_cleaned['customer_unique_id'].nunique()",
        "Value": total_customers
    },
    {
        "KPI": "Customer Repeat Rate (%)",
        "Grain": "Customer-level",
        "Base Population": "Customers with more than 1 order",
        "Formula": "(order_count > 1).sum() / total_customers * 100",
        "Value": repeat_customers_rate
    },
    {
        "KPI": "On-Time Delivery Rate (%)",
        "Grain": "Order-level",
        "Base Population": "Delivered orders only (order_delivered_customer_date IS NOT NULL)",
        "Formula": "is_on_time.mean() * 100",
        "Value": on_time_rate
    },
    {
        "KPI": "Late Delivery Rate (%)",
        "Grain": "Order-level",
        "Base Population": "Delivered orders with valid duration (excludes has_invalid_duration)",
        "Formula": "is_late.mean() * 100",
        "Value": is_late_rate
    },
    {
        "KPI": "Avg. Delivery Time - Mean (days)",
        "Grain": "Order-level",
        "Base Population": "Delivered orders with valid duration",
        "Formula": "purchase_to_customer_hours.mean() / 24",
        "Value": avg_delivery_hours / 24
    },
    {
        "KPI": "Avg. Delivery Time - Median (days)",
        "Grain": "Order-level",
        "Base Population": "Delivered orders with valid duration",
        "Formula": "purchase_to_customer_hours.median() / 24",
        "Value": median_delivery_hours / 24
    },
    {
        "KPI": "Average Review Score",
        "Grain": "Order-level (deduplicated, latest review kept)",
        "Base Population": "Orders with at least one review (~99.2% of all orders)",
        "Formula": "orders_master['review_score'].mean()",
        "Value": avg_review_score
    },
    {
        "KPI": "Low Rating Rate 1-2★ (%)",
        "Grain": "Order-level",
        "Base Population": "Same population as Average Review Score",
        "Formula": "review_score.le(2).mean() * 100",
        "Value": low_rating_rate
    },
    {
        "KPI": "Total Sellers",
        "Grain": "Seller-level",
        "Base Population": "All active sellers (at least 1 item sold)",
        "Formula": "items_master['seller_id'].nunique()",
        "Value": total_seller
    },
    {
        "KPI": "Seller Concentration Top 10% (%)",
        "Grain": "Seller-level",
        "Base Population": "Top 10% of sellers ranked by revenue",
        "Formula": "top_10pct_revenue.sum() / total_revenue * 100",
        "Value": top10pct_share
    },
]

kpi_df = pd.DataFrame(kpi_dictionary)
kpi_df['Value'] = kpi_df['Value'].round(2)
kpi_df

,KPI,Grain,Base Population,Formula,Value
0,Total Orders,Order-level,"All orders, no filter",orders_master['order_id'].nunique(),99441.00
1,Total Items Sold,Item-level,All items sold,items_master.shape[0],112650.00
2,Total Sales (R$),Order-level (aggregated from item),Sum of total_item_price across all orders,orders_master['total_item_price'].sum(),13591643.70
3,Average Order Value (R$),Order-level,All orders,orders_master['total_item_price'].mean(),137.75
4,AOV incl. Freight (R$),Order-level,All orders,(total_item_price + total_freight_value).mean(),160.58
5,Total Unique Customers,Customer-level (customer_unique_id),All unique customers (not per-order ID),customers_cleaned['customer_unique_id'].nunique(),96096.00
6,Customer Repeat Rate (%),Customer-level,Customers with more than 1 order,(order_count > 1).sum() / total_customers * 100,3.12
7,On-Time Delivery Rate (%),Order-level,Delivered orders only (order_delivered_custome...,is_on_time.mean() * 100,91.89
8,Late Delivery Rate (%),Order-level,Delivered orders with valid duration (excludes...,is_late.mean() * 100,6.85
9,Avg. Delivery Time - Mean (days),Order-level,Delivered orders with valid duration,purchase_to_customer_hours.mean() / 24,12.56
